In [6]:
from pathlib import Path

FILENAME = "Public_finances_Covid_-_TaxLab_data.xlsx"

def find_file(name: str, levels: int = 4) -> Path:
    start = Path.cwd().resolve()
    for folder in [start, *list(start.parents)[:levels]]:
        hits = list(folder.rglob(name))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Couldn't find {name} from {start} or its parents")

PATH = find_file(FILENAME)
print(PATH)  # check it's the right copy

/Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Peterson Blogs/data/Public_finances_Covid_-_TaxLab_data.xlsx


In [22]:
from pathlib import Path

import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---------- data path: ../data first, else search up the folder tree ----------
FILENAME = "Public_finances_Covid_-_TaxLab_data.xlsx"
PATH = Path("../data") / FILENAME
if not PATH.exists():
    PATH = next(
        p for parent in [Path.cwd(), *list(Path.cwd().parents)[:4]]
        for p in parent.rglob(FILENAME)
    )
print("Using:", PATH.resolve())

# ---------- settings ----------
GREY = "#676A86"
HEIGHT = 270

# ---------- data ----------
df = (
    pd.read_excel(
        PATH, sheet_name="Graph - LR debt", header=None,
        usecols="B:C", names=["year", "debt"],
    )
    .dropna()
    .astype({"year": int})
)
df["date"] = pd.to_datetime(df["year"], format="%Y")
df["series"] = "UK public debt"   # constant field so the theme assigns the colour

# war periods (Napoleonic span = French Revolutionary + Napoleonic Wars; use 1803 for the narrow definition)
wars = pd.DataFrame({
    "start": [1793, 1914, 1939],
    "end":   [1815, 1918, 1945],
})
wars["start"] = pd.to_datetime(wars["start"], format="%Y")
wars["end"] = pd.to_datetime(wars["end"], format="%Y")

# ---------- chart ----------
bands = alt.Chart(wars).mark_rect(color=GREY, opacity=0.12).encode(
    x="start:T", x2="end:T",
)

line = alt.Chart(df).mark_line(strokeWidth=1.8).encode(
    x=alt.X(
        "date:T", title=None,
        axis=alt.Axis(format="%Y", tickCount=alt.TimeIntervalStep(interval="year", step=50)),
    ),
    y=alt.Y(
        "debt:Q", scale=alt.Scale(zero=True),
        title="Public debt, share of GDP",
        axis=alt.Axis(labelExpr="datum.label + '%'"),
    ),
    color=alt.Color("series:N", legend=None),
)

caption = alt.Title(
    text="Source: Bank of England (1700–1899); OBR (1900–2025)",
    subtitle=["UK public debt as a share of GDP, %. Annual, 1700–2025. "
              "Shaded bands mark the Napoleonic Wars and the two World Wars."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color=GREY, subtitleColor=GREY, dy=12)

chart = (
    (bands + line)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".", name="public_debt_three_centuries", svg=True)
chart

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Peterson Blogs/data/Public_finances_Covid_-_TaxLab_data.xlsx


alt.LayerChart(...)